# 5. Memory / Conversation

Multi-turn conversation using `RunnableWithMessageHistory`: a chain gains
"memory" by attaching a `get_history(session_id)` function that returns a
`BaseChatMessageHistory` object per session — the chain reads/appends messages
to it automatically on every `.invoke()`.

This notebook keeps history in a plain in-process dict, the simplest possible
backend. The actual app additionally supports Redis and Postgres-backed history
(see `docs/langchain/05-memory-conversation.md`) — same interface, durable storage.

**Prerequisites:** Ollama running locally with `llama3.2` pulled.

### Setup

This cell makes the project's shared `tools`/`models` packages importable
regardless of where Jupyter's working directory actually is (it's usually
this notebook's own folder, not the repo root), and loads `.env` plus any
cached secrets in `.env.local` (populated by `scripts/lib/env.sh` the first
time you've run `scripts/start_app.sh` / `scripts/start_infra.sh`).

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
load_dotenv(project_root / ".env.local", override=True)  # cached secrets, if resolve_env() has run at least once
print("Project root on sys.path:", project_root)

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

from models.chat_models.ollama_models import SupportedModel, get_chat_model

llm = get_chat_model(SupportedModel.llama3_2)

# The "memory" itself: a plain dict mapping session_id -> history object.
store: dict[str, InMemoryChatMessageHistory] = {}


def get_history(session_id: str) -> InMemoryChatMessageHistory:
    return store.setdefault(session_id, InMemoryChatMessageHistory())


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant with a good memory for the current conversation."),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)

chain_with_history = RunnableWithMessageHistory(
    prompt | llm,
    get_history,
    input_messages_key="input",
    history_messages_key="history",
)

## Turn 1 — establish a fact

In [ ]:
session = {"configurable": {"session_id": "notebook-demo"}}

response_1 = chain_with_history.invoke({"input": "My favorite number is 42."}, config=session)
print(response_1.content)

## Turn 2 — same `session_id`, ask the model to recall it

In [ ]:
response_2 = chain_with_history.invoke({"input": "What is my favorite number?"}, config=session)
print(response_2.content)

In [ ]:
# The history object itself, for inspection:
for message in get_history("notebook-demo").messages:
    print(f"[{message.type}] {message.content}")

## 🧪 Playground

**1. A second, independent session** — use `session_id="other-session"` and ask the same recall question. It shouldn't know the favorite number.

In [ ]:
# TODO: invoke chain_with_history with a different session_id


**2. Clear a session's history** (`get_history("notebook-demo").clear()`) and confirm a follow-up recall question now fails.

In [ ]:
# TODO: clear the history and re-ask the recall question


**3. A longer conversation** — add 3-4 more turns to `"notebook-demo"` about different topics, then ask it to summarize what you've discussed.

In [ ]:
# TODO: add more turns, then ask for a summary
